### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys


sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Mon Jun 22 09:18:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   48C    P0             45W /  300W |       1MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/PassagePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/PassageDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
len(PC_faceage)

624

In [7]:
PC_faceage[0]

[(358, 286),
 (418, 400),
 (355, 460),
 (353, 188),
 (423, 89),
 (310, 16),
 (355, 270),
 (428, 404),
 (68, 459),
 (381, 8),
 (424, 87),
 (358, 175),
 (377, 24),
 (355, 207),
 (461, 386),
 (460, 417),
 (405, 85),
 (66, 433),
 (310, 111),
 (392, 364),
 (437, 426)]

In [8]:
df_faceage

,label,score
0,"a star. Our planet, Earth, orbits, or circles,...",1
1,"Adam, We did not have plastic toys. I played w...",1
2,Who said the little owl. Who wants to hunt wit...,1
3,dead leaf. This is a mole. Moles burrow underg...,1
4,ereaddatagradepsenvironcomp.html Environment r...,1
...,...,...
467,work over the summer on any changes they wish ...,12
468,between January and December plunged the Unite...,12
469,into a newly opened bank account. I was amazed...,12
470,"occurring phenomenon, manmade by products are ...",12


In [9]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)
classes = [0]*size

472


In [10]:
print(len(all_pc_faceage))

11763


In [11]:
crowd_labels = pd.read_csv('data/passage_cleaned.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [12]:
crowd_labels.head()

,Unnamed: 0,_unit_id,_created_at,_golden,_id,_missed,_started_at,_tainted,_trust,performer,please_decide_which_passage_is_more_difficult,label_level_a,label_level_b,left,right,please_decide_which_passage_is_more_difficult_gold,label
0,8371,101707741,9/22/2011 11:40,False,213246266,NaN,9/22/2011 11:36,True,0.5,5,Passage B is more difficult.,5,7,been wicked. They believed that the end of the...,lichen Sect Content Linking Artid A snake coil...,NaN,lichen Sect Content Linking Artid A snake coil...
1,10346,101707977,9/22/2011 11:40,False,213246261,NaN,9/22/2011 11:36,True,0.5,5,Passage A is more difficult.,9,8,primitive hunters surrounded the herds with fi...,"For one thing, in the second half of the thirt...",NaN,primitive hunters surrounded the herds with fi...
2,13442,101708316,9/22/2011 11:40,False,213246262,NaN,9/22/2011 11:36,True,0.5,5,Passage A is more difficult.,7,12,possible error HhexagonhistogramhypotenuseIIde...,while a simpler solution would be to use fuels...,NaN,possible error HhexagonhistogramhypotenuseIIde...
3,3673,101707232,9/22/2011 11:35,False,213244638,NaN,9/22/2011 11:32,True,0.5,5,Passage B is more difficult.,3,7,"to tell him how to go about catching fish. Oh,...",and homespun remedies to keep deer out of my g...,NaN,and homespun remedies to keep deer out of my g...
4,2004,101707077,9/22/2011 11:35,False,213244637,NaN,9/22/2011 11:32,True,0.5,5,Passage A is more difficult.,10,2,edict which prevented chaos was the following ...,cold it freezes. To freeze is to become very c...,NaN,edict which prevented chaos was the following ...


In [13]:
from pathlib import Path

# Map item -> score
score_dict = dict(zip(df_faceage["label"].apply(lambda x: Path(x).name),
                      df_faceage["score"]))

left_correct = 0
right_correct = 0
ties = 0
total = 0

for _, row in crowd_labels.iterrows():
    left = Path(row["left"]).name
    right = Path(row["right"]).name

    if left not in score_dict or right not in score_dict:
        print('here')
        continue  # skip if missing

    total += 1
    left_score = score_dict[left]
    right_score = score_dict[right]

    if left_score > right_score:
        left_correct += 1
    elif right_score > left_score:
        right_correct += 1
    else:
        ties += 1

print(f"Total comparisons: {total}")
print(f"Left is true winner: {left_correct} ({left_correct/total:.3f})")
print(f"Right is true winner: {right_correct} ({right_correct/total:.3f})")
print(f"Ties: {ties} ({ties/total:.3f})")

Total comparisons: 11763
Left is true winner: 5604 (0.476)
Right is true winner: 6159 (0.524)
Ties: 0 (0.000)


In [12]:
# PC_faceage

In [23]:
from collections import defaultdict, Counter

# Example input:
# data = {
#     0: {(1, 2), (2, 3), (3, 4)},
#     1: {(1, 2), (2, 3), (3, 1)},   # cycle
#     2: {(5, 6), (7, 8)}            # acyclic
# }

def is_acyclic_undirected(comparisons):
    """
    comparisons can contain duplicates like:
    [(1,2), (1,2), (2,3)]

    Duplicate edges are collapsed to one edge only.
    Self-loops count as cycles.
    """

    # Count duplicates if needed
    edge_counts = Counter(comparisons)

    # Collapse duplicates + treat as undirected
    unique_edges = set()

    for u, v in edge_counts:
        if u == v:
            return False  # self-loop => cycle

        edge = (min(u, v), max(u, v))
        unique_edges.add(edge)

    graph = defaultdict(list)

    for u, v in unique_edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()

    def dfs(node, parent):
        visited.add(node)

        for nbr in graph[node]:
            if nbr not in visited:
                if not dfs(nbr, node):
                    return False
            elif nbr != parent:
                return False

        return True

    for node in graph:
        if node not in visited:
            if not dfs(node, -1):
                return False

    return True

def check_workers(data):
    """
    For each worker, check whether their comparison graph is acyclic.
    Returns dict: worker_id -> True/False
    """
    result = {}

    for worker_id, comparisons in data.items():
        result[worker_id] = is_acyclic_undirected(comparisons)

    return result


# Example usage
if __name__ == "__main__":
    data = {
        0: {(1, 2), (2, 3), (3, 4)},       # tree -> acyclic
        1: {(1, 2), (2, 3), (3, 1)},       # triangle -> cyclic
        2: {(5, 6), (7, 8)},               # disconnected forest -> acyclic
        3: {(1, 2), (2, 3), (3, 4), (4, 2)} # cycle
    }

    results = check_workers(PC_faceage)
    
    cnt = 0
    for worker, acyclic in results.items():
        if not acyclic:
#             cnt += 1
            print(f"Worker {worker}: {'Acyclic' if acyclic else 'Cyclic'}")
            
print("Done")
print(cnt == len(results.keys()))

Worker 6: Cyclic
Worker 178: Cyclic
Done
False


In [27]:
print(PC_faceage[178])

[(417, 351), (360, 136), (406, 294), (365, 408), (431, 452), (388, 87), (179, 265), (351, 213), (385, 236), (351, 4), (283, 243), (406, 438), (288, 208), (288, 47), (425, 460), (405, 85), (433, 66), (160, 253), (142, 406), (429, 222), (59, 471), (459, 68), (207, 355), (467, 268), (366, 272), (375, 67), (294, 438), (358, 468), (426, 24), (16, 199), (460, 426), (470, 202), (381, 333), (362, 286), (373, 458), (360, 332), (245, 294), (305, 248), (424, 440), (390, 440)]


In [29]:
print(len(PC_faceage))

624


In [28]:
import networkx as nx

edges = [(417, 351), (360, 136), (406, 294), (365, 408), (431, 452), (388, 87), (179, 265), (351, 213), (385, 236), (351, 4), (283, 243), (406, 438), (288, 208), (288, 47), (425, 460), (405, 85), (433, 66), (160, 253), (142, 406), (429, 222), (59, 471), (459, 68), (207, 355), (467, 268), (366, 272), (375, 67), (294, 438), (358, 468), (426, 24), (16, 199), (460, 426), (470, 202), (381, 333), (362, 286), (373, 458), (360, 332), (245, 294), (305, 248), (424, 440), (390, 440)]

G = nx.Graph()
G.add_edges_from(edges)

print(nx.is_forest(G))   # False means cyclic
print(nx.cycle_basis(G)) # lists cycles

False
[[406, 438, 294]]
